# Cocktail Creator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/cocktail-creator/blob/main/Cocktail_Creator.ipynb)

A fine-tuned T5 sequence-to-sequence model that generates complete cocktail recipes — ingredients, quantities, method, and glass — from one or more seed ingredient names.

## Introduction

The model is trained on real bartender recipes from the [`erwanlc/cocktails_recipe`](https://huggingface.co/datasets/erwanlc/cocktails_recipe) dataset. Each ingredient name is augmented with flavor descriptors sourced from [FlavorDB](https://cosylab.iiitd.edu.in/flavordb2/), which are embedded directly in this notebook to avoid any external API calls that have broken the notebook in the past.

The key idea behind the training format is that **seed ingredients carry no quantities**. This forces the model to predict appropriate quantities for every ingredient in the output, including the ones provided by the user of the model:

```
INPUT  : create cocktail with: bourbon [vanilla caramel oak]; lemon juice [sour citrus fresh]
OUTPUT : Ingredients: 4.5 cl bourbon, 2.25 cl lemon juice, 1.5 cl honey syrup, 2 dash angostura bitters | Method: shake and strain | Glass: coupe glass
```

---
## 1. Setup

First, let's confirm a GPU is available as training takes significant time of the model (between 60-120 minutes with A100, ~20x that using CPU).

In [1]:
import subprocess
import torch

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)

if r.returncode != 0:
    raise RuntimeError('No GPU found. Go to Runtime > Change runtime type > T4 GPU')

print('GPU:', r.stdout.strip())
print('PyTorch:', torch.__version__)
print('CUDA device:', torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB
PyTorch: 2.10.0+cu128
CUDA device: NVIDIA A100-SXM4-40GB


Install the packages we need.

- peft — implements LoRA, which lets us fine-tune only ~1% of T5's parameters
- sentencepiece — the tokeniser backend required by T5
- rapidfuzz — fast fuzzy string matching for the flavor lookup
- huggingface_hub — saves the trained model to Jovin's HF Hub account so it persists between sessions and is accessible by all

In [2]:
!pip install -q peft==0.10.0 accelerate sentencepiece datasets rapidfuzz huggingface_hub
print('Done.')

Done.


We'll save the trained model to [Hugging Face Hub](https://huggingface.co) rather than Google Drive. This means the model lives at a permanent public URL (`hf.co/jovinkk/cocktail-creator-t5`) and can be loaded in one line on any machine — including Kaggle, a local laptop, or a fresh Colab session, without any Drive mounting or auth popups.

In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN    = userdata.get('HF_TOKEN')
HF_REPO     = 'jovinkk/cocktail-creator-t5'

login(token=HF_TOKEN)
print(f'Logged in. Model will be pushed to: https://huggingface.co/{HF_REPO}')

Logged in. Model will be pushed to: https://huggingface.co/jovinkk/cocktail-creator-t5


---
## 2. Flavor Profile Database

T5 is a pure text model — it has no built-in understanding that `gin` is herbal and piney while `rum` is sweet and tropical. Without extra context, these two words look nearly identical to the model (both are short, common tokens with no structural difference).

We solve this by appending flavor tags to every seed ingredient. The tags come from a curated map of ~160 ingredients derived from FlavorDB. The function `get_flavor()` tries four strategies in order: exact lookup → fuzzy match → substring → keyword fallback.

In [4]:
from rapidfuzz import process as fuzz_process

FLAVOR_DB = {
    # ── Base spirits ──────────────────────────────────────────────────────────
    'gin':                   ['herbal', 'floral', 'citrus', 'piney', 'botanical'],
    'london dry gin':        ['juniper', 'citrus', 'dry', 'herbal', 'crisp'],
    'sloe gin':              ['berry', 'sweet', 'almond', 'bitter', 'fruity'],
    'vodka':                 ['neutral', 'clean', 'ethanol', 'mild'],
    'citrus vodka':          ['citrus', 'clean', 'bright', 'light'],
    'rum':                   ['sweet', 'caramel', 'tropical', 'molasses'],
    'light rum':             ['sweet', 'clean', 'light', 'tropical'],
    'white rum':             ['sweet', 'clean', 'light', 'tropical'],
    'dark rum':              ['molasses', 'caramel', 'smoky', 'vanilla', 'rich'],
    'aged rum':              ['caramel', 'vanilla', 'oak', 'tropical', 'sweet'],
    'spiced rum':            ['spicy', 'vanilla', 'caramel', 'warm', 'sweet'],
    'bourbon':               ['vanilla', 'caramel', 'oak', 'sweet', 'woody', 'corn'],
    'whiskey':               ['woody', 'malty', 'grain', 'sweet', 'warm'],
    'rye whiskey':           ['spicy', 'peppery', 'grain', 'woody', 'dry'],
    'scotch':                ['smoky', 'peaty', 'malt', 'woody', 'complex'],
    'scotch whisky':         ['smoky', 'peaty', 'malt', 'woody', 'complex'],
    'blended scotch':        ['smooth', 'malty', 'light', 'woody', 'smoky'],
    'irish whiskey':         ['smooth', 'light', 'malty', 'floral', 'clean'],
    'japanese whisky':       ['delicate', 'floral', 'malty', 'smooth', 'oak'],
    'tequila':               ['earthy', 'citrus', 'agave', 'herbaceous', 'peppery'],
    'blanco tequila':        ['earthy', 'citrus', 'agave', 'fresh', 'peppery'],
    'silver tequila':        ['earthy', 'citrus', 'agave', 'fresh', 'clean'],
    'reposado tequila':      ['earthy', 'agave', 'vanilla', 'oak', 'spicy'],
    'anejo tequila':         ['oak', 'vanilla', 'caramel', 'agave', 'complex'],
    'mezcal':                ['smoky', 'earthy', 'agave', 'complex', 'woody'],
    'brandy':                ['fruity', 'sweet', 'vanilla', 'oak', 'grape'],
    'cognac':                ['fruity', 'floral', 'oak', 'vanilla', 'sweet', 'grape'],
    'pisco':                 ['fruity', 'floral', 'grape', 'fresh', 'clean'],
    'cachaca':               ['grassy', 'sweet', 'tropical', 'cane', 'funky'],
    'absinthe':              ['anise', 'herbal', 'bitter', 'fennel', 'complex'],
    'aquavit':               ['caraway', 'herbal', 'anise', 'spicy', 'clean'],
    'calvados':              ['apple', 'fruity', 'oak', 'caramel', 'sweet'],
    'grappa':                ['grape', 'floral', 'fruity', 'sharp', 'complex'],
    # ── Liqueurs and fortified wines ──────────────────────────────────────────
    'amaretto':              ['almond', 'sweet', 'marzipan', 'stone fruit', 'warm'],
    'kahlua':                ['coffee', 'sweet', 'vanilla', 'chocolate', 'rum'],
    'coffee liqueur':        ['coffee', 'sweet', 'vanilla', 'bitter', 'rich'],
    'triple sec':            ['citrus', 'orange', 'sweet', 'bright'],
    'cointreau':             ['citrus', 'orange', 'sweet', 'bitter', 'aromatic'],
    'grand marnier':         ['orange', 'cognac', 'sweet', 'complex', 'warm'],
    'campari':               ['bitter', 'herbal', 'citrus', 'spicy', 'complex'],
    'aperol':                ['bitter', 'sweet', 'citrus', 'herbal', 'light'],
    'frangelico':            ['hazelnut', 'sweet', 'nutty', 'warm', 'toasted'],
    'st germain':            ['floral', 'sweet', 'delicate', 'fruity', 'elderflower'],
    'elderflower liqueur':   ['floral', 'sweet', 'delicate', 'fruity', 'fragrant'],
    'chambord':              ['raspberry', 'fruity', 'sweet', 'berry', 'floral'],
    'maraschino':            ['cherry', 'almond', 'sweet', 'nutty', 'floral'],
    'maraschino liqueur':    ['cherry', 'almond', 'sweet', 'nutty', 'floral'],
    'creme de cacao':        ['chocolate', 'sweet', 'vanilla', 'cocoa'],
    'creme de menthe':       ['mint', 'sweet', 'fresh', 'cool'],
    'green chartreuse':      ['herbal', 'complex', 'spicy', 'sweet', 'botanical'],
    'yellow chartreuse':     ['herbal', 'honey', 'sweet', 'mild', 'botanical'],
    'chartreuse':            ['herbal', 'complex', 'sweet', 'spicy', 'botanical'],
    'baileys':               ['cream', 'chocolate', 'sweet', 'dairy', 'vanilla'],
    'drambuie':              ['honey', 'herbal', 'scotch', 'sweet', 'spiced'],
    'benedictine':           ['herbal', 'honey', 'complex', 'sweet', 'botanical'],
    'amaro':                 ['bitter', 'herbal', 'complex', 'sweet', 'aromatic'],
    'amaro nonino':          ['bitter', 'fruity', 'herbal', 'sweet', 'orange'],
    'fernet branca':         ['bitter', 'herbal', 'minty', 'complex', 'medicinal'],
    'cynar':                 ['bitter', 'earthy', 'herbal', 'artichoke'],
    'limoncello':            ['lemon', 'citrus', 'sweet', 'fresh', 'zesty'],
    'peach schnapps':        ['peach', 'sweet', 'fruity', 'stone fruit'],
    'raspberry liqueur':     ['raspberry', 'sweet', 'fruity', 'berry'],
    'blackberry liqueur':    ['blackberry', 'sweet', 'fruity', 'berry'],
    'blue curacao':          ['citrus', 'orange', 'sweet', 'tropical'],
    'falernum':              ['almond', 'lime', 'clove', 'spicy', 'sweet'],
    'orgeat':                ['almond', 'sweet', 'nutty', 'floral', 'creamy'],
    'sweet vermouth':        ['sweet', 'herbal', 'spicy', 'fruity', 'wine'],
    'dry vermouth':          ['dry', 'herbal', 'floral', 'crisp', 'wine'],
    'lillet blanc':          ['floral', 'fruity', 'citrus', 'wine', 'sweet'],
    'lillet rose':           ['fruity', 'floral', 'rose', 'wine', 'delicate'],
    'prosecco':              ['fruity', 'light', 'bubbly', 'crisp', 'pear'],
    'champagne':             ['yeasty', 'crisp', 'fruity', 'bubbly', 'floral'],
    'sparkling wine':        ['crisp', 'fruity', 'bubbly', 'light', 'dry'],
    'port':                  ['sweet', 'fruity', 'rich', 'fortified', 'plum'],
    'sherry':                ['nutty', 'dry', 'complex', 'oxidised', 'wine'],
    # ── Juices ────────────────────────────────────────────────────────────────
    'lemon juice':           ['sour', 'citrus', 'tart', 'fresh', 'bright'],
    'fresh lemon juice':     ['sour', 'citrus', 'tart', 'fresh', 'bright'],
    'lime juice':            ['sour', 'citrus', 'tart', 'tropical', 'fresh'],
    'fresh lime juice':      ['sour', 'citrus', 'tart', 'tropical', 'fresh'],
    'orange juice':          ['sweet', 'citrus', 'fruity', 'fresh', 'bright'],
    'fresh orange juice':    ['sweet', 'citrus', 'fruity', 'fresh', 'bright'],
    'grapefruit juice':      ['bitter', 'citrus', 'tart', 'fresh', 'sour'],
    'pineapple juice':       ['sweet', 'tropical', 'fruity', 'tart'],
    'cranberry juice':       ['tart', 'fruity', 'berry', 'sour', 'sweet'],
    'apple juice':           ['sweet', 'fruity', 'fresh', 'apple', 'light'],
    'pomegranate juice':     ['tart', 'fruity', 'sweet', 'berry', 'complex'],
    'passion fruit juice':   ['tropical', 'tart', 'fruity', 'exotic', 'sweet'],
    'mango juice':           ['tropical', 'sweet', 'fruity', 'thick', 'rich'],
    # ── Syrups and sweeteners ─────────────────────────────────────────────────
    'simple syrup':          ['sweet', 'neutral', 'clean'],
    'honey syrup':           ['sweet', 'floral', 'honey', 'rich'],
    'honey':                 ['sweet', 'floral', 'rich', 'complex', 'natural'],
    'agave syrup':           ['sweet', 'mild', 'neutral', 'clean'],
    'agave nectar':          ['sweet', 'mild', 'neutral', 'clean'],
    'grenadine':             ['sweet', 'pomegranate', 'fruity', 'floral'],
    'raspberry syrup':       ['sweet', 'fruity', 'berry', 'bright'],
    'passion fruit syrup':   ['tropical', 'sour', 'fruity', 'exotic', 'sweet'],
    'elderflower cordial':   ['floral', 'sweet', 'delicate', 'elderflower'],
    'ginger syrup':          ['spicy', 'sweet', 'ginger', 'warm'],
    'cinnamon syrup':        ['spicy', 'sweet', 'warm', 'woody'],
    'lavender syrup':        ['floral', 'sweet', 'aromatic', 'delicate'],
    'vanilla syrup':         ['sweet', 'vanilla', 'creamy', 'warm'],
    'maple syrup':           ['sweet', 'woody', 'caramel', 'maple', 'rich'],
    # ── Carbonated mixers ─────────────────────────────────────────────────────
    'ginger beer':           ['spicy', 'ginger', 'sweet', 'carbonated', 'sharp'],
    'ginger ale':            ['light', 'ginger', 'sweet', 'carbonated', 'mild'],
    'club soda':             ['neutral', 'carbonated', 'crisp', 'clean'],
    'soda water':            ['neutral', 'carbonated', 'crisp', 'clean'],
    'tonic water':           ['bitter', 'quinine', 'carbonated', 'crisp'],
    'cola':                  ['sweet', 'caramel', 'cola', 'carbonated'],
    'lemonade':              ['sour', 'sweet', 'citrus', 'fresh'],
    # ── Creams and dairy ──────────────────────────────────────────────────────
    'coconut cream':         ['sweet', 'tropical', 'creamy', 'coconut', 'rich'],
    'coconut water':         ['fresh', 'sweet', 'light', 'tropical', 'clean'],
    'heavy cream':           ['rich', 'creamy', 'dairy', 'fat', 'smooth'],
    'double cream':          ['rich', 'creamy', 'dairy', 'fat', 'smooth'],
    'half and half':         ['light', 'creamy', 'dairy', 'mild'],
    'milk':                  ['light', 'creamy', 'dairy', 'mild', 'neutral'],
    'egg white':             ['protein', 'frothy', 'creamy', 'neutral'],
    'whole egg':             ['rich', 'creamy', 'protein', 'neutral'],
    # ── Coffee ────────────────────────────────────────────────────────────────
    'cold brew coffee':      ['coffee', 'bitter', 'dark', 'smooth', 'roasted'],
    'espresso':              ['coffee', 'bitter', 'intense', 'roasted', 'dark'],
    'cold brew':             ['coffee', 'bitter', 'dark', 'smooth', 'roasted'],
    'tea':                   ['tannin', 'floral', 'earthy', 'mild', 'aromatic'],
    'matcha':                ['earthy', 'grassy', 'bitter', 'umami', 'vegetal'],
    # ── Bitters ───────────────────────────────────────────────────────────────
    'angostura bitters':     ['bitter', 'spicy', 'herbal', 'aromatic', 'complex'],
    "peychaud's bitters":    ['bitter', 'floral', 'anise', 'light', 'aromatic'],
    'orange bitters':        ['bitter', 'citrus', 'orange', 'spicy', 'dry'],
    'chocolate bitters':     ['bitter', 'chocolate', 'cocoa', 'spicy', 'complex'],
    'mole bitters':          ['bitter', 'chocolate', 'spicy', 'chili', 'complex'],
    'aromatic bitters':      ['bitter', 'spicy', 'herbal', 'aromatic'],
    # ── Fresh herbs and produce ───────────────────────────────────────────────
    'mint':                  ['mint', 'fresh', 'cool', 'herbal', 'menthol'],
    'fresh mint':            ['mint', 'fresh', 'cool', 'herbal', 'menthol'],
    'basil':                 ['herbal', 'sweet', 'peppery', 'floral', 'fresh'],
    'cucumber':              ['fresh', 'green', 'crisp', 'light', 'cool'],
    'rosemary':              ['herbal', 'piney', 'floral', 'woody', 'aromatic'],
    'thyme':                 ['herbal', 'earthy', 'minty', 'aromatic'],
    'jalapeño':              ['spicy', 'vegetal', 'fresh', 'hot'],
    'ginger':                ['spicy', 'warm', 'pungent', 'fresh', 'aromatic'],
    'fresh ginger':          ['spicy', 'warm', 'pungent', 'fresh', 'aromatic'],
    'cinnamon':              ['spicy', 'warm', 'sweet', 'woody', 'aromatic'],
    'nutmeg':                ['spicy', 'warm', 'sweet', 'woody', 'nutty'],
    'vanilla':               ['sweet', 'creamy', 'floral', 'warm', 'rich'],
    'cardamom':              ['spicy', 'aromatic', 'floral', 'warm', 'complex'],
    # ── Fruits ────────────────────────────────────────────────────────────────
    'lemon':                 ['sour', 'citrus', 'bright', 'zesty', 'fresh'],
    'lime':                  ['sour', 'citrus', 'tart', 'tropical', 'fresh'],
    'orange':                ['sweet', 'citrus', 'fruity', 'bright', 'zesty'],
    'grapefruit':            ['bitter', 'citrus', 'tart', 'fresh'],
    'peach':                 ['sweet', 'fruity', 'stone fruit', 'floral', 'soft'],
    'strawberry':            ['sweet', 'fruity', 'berry', 'bright'],
    'raspberry':             ['tart', 'sweet', 'fruity', 'berry'],
    'blackberry':            ['tart', 'sweet', 'fruity', 'berry', 'earthy'],
    'blueberry':             ['sweet', 'fruity', 'berry', 'mild'],
    'watermelon':            ['sweet', 'fresh', 'light', 'fruity'],
    'cherry':                ['sweet', 'fruity', 'tart', 'stone fruit'],
    'mango':                 ['tropical', 'sweet', 'fruity', 'rich'],
    'pineapple':             ['tropical', 'sweet', 'tart', 'fruity'],
    'passion fruit':         ['tropical', 'tart', 'exotic', 'fruity'],
    'coconut':               ['sweet', 'tropical', 'creamy', 'rich'],
    'apple':                 ['sweet', 'fruity', 'crisp', 'fresh', 'tart'],
}

_FLAVOR_KEYS = list(FLAVOR_DB.keys())

# Keyword fallbacks for brand names not in the dictionary (e.g. 'Diplomatico Rum')
_KW_FALLBACKS = [
    ('whisky',   ['woody', 'smoky', 'vanilla', 'caramel']),
    ('whiskey',  ['woody', 'smoky', 'vanilla', 'caramel']),
    ('bourbon',  ['vanilla', 'caramel', 'oak', 'sweet']),
    ('rum',      ['sweet', 'caramel', 'tropical', 'molasses']),
    ('gin',      ['herbal', 'citrus', 'floral', 'botanical']),
    ('vodka',    ['neutral', 'clean', 'ethanol', 'mild']),
    ('tequila',  ['earthy', 'citrus', 'agave', 'herbaceous']),
    ('mezcal',   ['smoky', 'earthy', 'agave', 'complex']),
    ('brandy',   ['fruity', 'oak', 'vanilla', 'grape']),
    ('cognac',   ['fruity', 'floral', 'oak', 'vanilla']),
    ('juice',    ['fresh', 'fruity', 'sour', 'bright']),
    ('syrup',    ['sweet', 'rich', 'neutral']),
    ('bitters',  ['bitter', 'aromatic', 'herbal', 'complex']),
    ('cream',    ['creamy', 'dairy', 'rich', 'smooth']),
    ('liqueur',  ['sweet', 'aromatic', 'complex']),
    ('vermouth', ['herbal', 'wine', 'aromatic']),
    ('soda',     ['carbonated', 'neutral', 'crisp']),
    ('wine',     ['fruity', 'tannin', 'dry', 'grape']),
    ('mint',     ['mint', 'fresh', 'cool', 'herbal']),
    ('ginger',   ['spicy', 'warm', 'pungent', 'fresh']),
    ('coffee',   ['coffee', 'bitter', 'dark', 'roasted']),
    ('honey',    ['sweet', 'floral', 'rich', 'natural']),
]

def get_flavor(name: str) -> list:
    """Return flavor descriptors for an ingredient name.

    Tries in order: exact match → fuzzy match (≥82% similarity) →
    substring match → keyword fallback → default 'aromatic'.
    """
    key = name.lower().strip()
    if key in FLAVOR_DB:
        return FLAVOR_DB[key][:5]
    hit = fuzz_process.extractOne(key, _FLAVOR_KEYS, score_cutoff=82)
    if hit:
        return FLAVOR_DB[hit[0]][:5]
    for k, v in FLAVOR_DB.items():
        if k in key or key in k:
            return v[:5]
    for kw, flv in _KW_FALLBACKS:
        if kw in key:
            return flv
    return ['aromatic']


print(f'Loaded {len(FLAVOR_DB)} flavor profiles.')
print('Sample lookups:')
for s in ['gin', 'dark rum', 'campari', 'lemon juice', 'Diplomatico Rum']:
    print(f'  {s:25s} → {get_flavor(s)}')

Loaded 157 flavor profiles.
Sample lookups:
  gin                       → ['herbal', 'floral', 'citrus', 'piney', 'botanical']
  dark rum                  → ['molasses', 'caramel', 'smoky', 'vanilla', 'rich']
  campari                   → ['bitter', 'herbal', 'citrus', 'spicy', 'complex']
  lemon juice               → ['sour', 'citrus', 'tart', 'fresh', 'bright']
  Diplomatico Rum           → ['sweet', 'caramel', 'tropical', 'molasses']


---
## 3. Load and Prepare the Dataset

We use the `erwanlc/cocktails_recipe` dataset from Hugging Face — real bartender recipes with structured ingredient lists, preparation method, and glass type.

The parsing pipeline has one important design decision: **quantities are stripped from the input but kept in the output**. If quantities appeared in the input, the model would treat them as fixed and only predict quantities for the remaining ingredients. By removing them entirely, the model must learn to produce appropriate quantities for every ingredient — including the seeds the user provides at inference time.

> **Design Choice — recipe-level split:** We divide the data into train/test at the recipe level *before* building training pairs. If we split at the pair level, the same recipe could appear in both train and test with different seeds — the model would be evaluated on cocktails it had already memorised. Recipe-level splitting guarantees a clean held-out set.

In [5]:
import re, json, random, ast
from pathlib import Path
from datasets import load_dataset

random.seed(42)
Path('data').mkdir(exist_ok=True)

print('Loading erwanlc/cocktails_recipe ...')
raw_ds = load_dataset('erwanlc/cocktails_recipe', split='train')
print(f'Loaded {len(raw_ds)} recipes | columns: {raw_ds.column_names}')
print('\nFirst record:')
for k, v in raw_ds[0].items():
    print(f'  {k:15s}: {repr(v)[:120]}')

Loading erwanlc/cocktails_recipe ...
Loaded 6956 recipes | columns: ['title', 'glass', 'garnish', 'recipe', 'ingredients']

First record:
  title          : 'Abacaxi Ricaço'
  glass          : 'Pineapple shell (frozen) glass'
  garnish        : 'Cut a straw sized hole in the top of the pineapple shell & replace it as a lid'
  recipe         : 'Cut the top off a small pineapple and carefully scoop out the flesh from the base to leave a shell with 12mm (½ inch) t
  ingredients    : "[['1 whole', 'Pineapple (fresh)'], ['9 cl', 'Havana Club 3 Year Old rum'], ['2.25 cl', 'Lime juice (freshly squeezed)']


The `ingredients` column is stored as a Python `repr` string with single quotes (e.g. `"[['4.5 cl', 'Gin']]"`) so we use `ast.literal_eval` rather than `json.loads`.

In [6]:
# Regex to extract a leading quantity+unit from an ingredient string
# e.g. "4.5 cl gin" → qty="4.5 cl", name="gin"
_QTY_RE = re.compile(
    r'^([\d\u00bc\u00bd\u00be\u2153\u2154]+(?:[\/\.]\d+)?\s*'
    r'(?:oz|cl|ml|tsp|tbsp|teaspoon|tablespoon|cup|part|parts|'
    r'dash(?:es)?|drop|splash(?:es)?|pinch|barspoon|jigger|shot|scoop|measure)s?'
    r'(?:\s+of)?)\s*(.+)$',
    re.IGNORECASE,
)

def _parse_ingredient_line(text):
    """Parse one ingredient string into {'qty': ..., 'name': ...}."""
    text = re.sub(r'^[-\u2022*\u00b7\s]+', '', str(text)).strip()
    if not text or len(text) < 2:
        return None
    m = _QTY_RE.match(text)
    if m:
        qty, name = m.group(1).strip(), m.group(2).strip().lower()
        return {'qty': qty, 'name': name} if name else None
    m2 = re.match(r'^(\d+(?:[\/\.]\d+)?)\s+(\S.+)$', text)
    if m2:
        return {'qty': m2.group(1), 'name': m2.group(2).lower()}
    return {'qty': '', 'name': text.lower()}


def parse_recipe(rec):
    """Convert a raw HuggingFace record into a clean dict, or return None."""
    glass  = (rec.get('glass')  or '').strip().lower()
    method = (rec.get('recipe') or '').strip()
    raw_ings = rec.get('ingredients', None)

    if isinstance(raw_ings, str):
        try:
            raw_ings = ast.literal_eval(raw_ings)
        except Exception:
            try:
                raw_ings = json.loads(raw_ings)
            except Exception:
                raw_ings = [x.strip() for x in raw_ings.split('\n') if x.strip()]

    if not isinstance(raw_ings, list):
        return None

    ings = []
    for item in raw_ings:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            qty, name = str(item[0]).strip(), str(item[1]).strip().lower()
            if name:
                ings.append({'qty': qty, 'name': name})
        elif isinstance(item, str) and item.strip():
            p = _parse_ingredient_line(item)
            if p and p['name']:
                ings.append(p)

    if len(ings) < 2:
        return None

    # Truncate method to first sentence to keep output targets short and consistent
    method_short = re.split(r'(?<=[.!?])\s+', method)[0][:140] if method else 'shake and strain'

    return {'ingredients': ings, 'method': method_short, 'glass': glass or 'cocktail glass'}


# Training pair builder ---------------------------------------------------------
# For each recipe we create 2-3 pairs by sampling 1, 2, or 3 seed ingredients.
# This teaches the model to work from any number of starting ingredients.

def _fmt_seed(ing):
    """Format one ingredient for the INPUT: name only + flavor tags, no quantity."""
    flavors = get_flavor(ing['name'])[:3]
    return f"{ing['name']} [{' '.join(flavors)}]"

def _fmt_output_ingredient(ing):
    """Format one ingredient for the OUTPUT: quantity + name."""
    qty = (ing['qty'] + ' ') if ing['qty'] else ''
    return qty + ing['name']

def recipe_to_pairs(rec, rng):
    """Generate training pairs from one recipe by sampling different seed subsets."""
    ings        = rec['ingredients']
    n           = len(ings)
    seed_counts = [1, 2] if n <= 3 else [1, 2, 3]

    full_ings = ', '.join(_fmt_output_ingredient(i) for i in ings)
    output    = (f"Ingredients: {full_ings} | Method: {rec['method']} | Glass: {rec['glass']}")

    pairs = []
    for k in seed_counts:
        if k > n:
            continue
        seeds    = rng.sample(ings, k)
        seed_str = '; '.join(_fmt_seed(s) for s in seeds)
        pairs.append({'input': f'create cocktail with: {seed_str}', 'output': output})
    return pairs

In [7]:
# Parse and split ---------------------------------------------------------------
print('Parsing recipes ...')
rng = random.Random(42)
all_recipes = [parse_recipe(r) for r in raw_ds]
all_recipes = [r for r in all_recipes if r is not None]
print(f'  Usable: {len(all_recipes):,} / {len(raw_ds):,}')

rng.shuffle(all_recipes)
cut           = int(0.8 * len(all_recipes))
train_recipes = all_recipes[:cut]
test_recipes  = all_recipes[cut:]
print(f'  Train recipes: {len(train_recipes)} | Test recipes: {len(test_recipes)}')

# Build pairs from training recipes only
all_train = []
for r in train_recipes:
    all_train.extend(recipe_to_pairs(r, rng))
rng.shuffle(all_train)

val_cut     = int(0.9 * len(all_train))
train_pairs = all_train[:val_cut]
val_pairs   = all_train[val_cut:]

# One standardised pair per test recipe (single seed for comparability)
test_pairs = []
for r in test_recipes:
    seed   = rng.sample(r['ingredients'], 1)
    output = (f"Ingredients: {', '.join(_fmt_output_ingredient(i) for i in r['ingredients'])} | "
              f"Method: {r['method']} | Glass: {r['glass']}")
    test_pairs.append({'input': f'create cocktail with: {_fmt_seed(seed[0])}', 'output': output})

print(f'  Train pairs: {len(train_pairs):,} | Val pairs: {len(val_pairs):,} | Test pairs: {len(test_pairs):,}')

for split, data in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]:
    with open(f'data/{split}.jsonl', 'w') as f:
        for p in data:
            f.write(json.dumps(p) + '\n')

print('\nSample pair:')
print(f'  INPUT : {train_pairs[0]["input"]}')
print(f'  OUTPUT: {train_pairs[0]["output"][:120]}...')

Parsing recipes ...
  Usable: 6,952 / 6,956
  Train recipes: 5561 | Test recipes: 1391
  Train pairs: 14,078 | Val pairs: 1,565 | Test pairs: 1,391

Sample pair:
  INPUT : create cocktail with: canadian icewine [fruity tannin dry]; martini extra dry vermouth [dry herbal floral]
  OUTPUT: Ingredients: 6 cl ketel one vodka, 1.5 cl martini extra dry vermouth, 2.25 cl canadian icewine | Method: STIR all ingred...


---
## 4. Fine-Tuning T5 with LoRA

**T5** (Text-to-Text Transfer Transformer) frames every NLP task as "text in → text out", making it a natural fit for recipe generation. We use the `base` variant (220M parameters).

**LoRA** (Low-Rank Adaptation) avoids updating all 220M parameters. Instead, it inserts small trainable matrices of rank `r` into the attention layers, freezing everything else. Only ~1.3M parameters are trained — less than 1% of the model — which cuts memory and time by roughly 10×.

**Early stopping** monitors validation loss every 500 steps and halts training if it does not improve for 5 consecutive checks. This prevents the model from **overfitting** — memorising training recipes rather than learning the general structure of a cocktail.

> **Design Choice — T5 over GPT-2:** This is a **sequence-to-sequence** task: we have a structured input (seed ingredients) and a structured output (full recipe). T5's encoder-decoder architecture is purpose-built for this. A decoder-only model like GPT-2 would need to concatenate input and output into a single stream, wasting context length on the input and making it harder to separate what was given from what was generated.

In [8]:
import torch
from datasets import load_dataset as ld
from transformers import (
    T5ForConditionalGeneration, T5Tokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from peft import get_peft_model, LoraConfig, TaskType

# Token length budget
# MAX_IN=160  fits  "create cocktail with: ingredient [tag tag]; ingredient [tag tag]..."
# MAX_OUT=200 fits  "Ingredients: ... | Method: ... | Glass: ..."
MAX_IN, MAX_OUT = 160, 200

print('Loading T5-base ...')
tokenizer = T5Tokenizer.from_pretrained('t5-base')
base      = T5ForConditionalGeneration.from_pretrained('t5-base')

# Apply LoRA to query (q) and value (v) projections in every attention layer.
# These two matrices have the most influence on what the model attends to and
# therefore what it generates, making them the highest-leverage targets.
model = get_peft_model(base, LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = 16,       # low-rank bottleneck dimension
    lora_alpha     = 32,       # scaling: rule of thumb is 2 × r
    target_modules = ['q', 'v'],
    lora_dropout   = 0.1,
))
model.print_trainable_parameters()  # should show ~0.8% trainable

Loading T5-base ...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7875765272113843


In [9]:
# Tokenise the data
ds = ld('json', data_files={'train': 'data/train.jsonl', 'validation': 'data/val.jsonl'})

def tokenize(batch):
    """Convert text pairs to tensors. Pad positions in labels are set to -100
    so PyTorch ignores them when computing the cross-entropy loss."""
    enc = tokenizer(batch['input'],  max_length=MAX_IN,  truncation=True, padding='max_length')
    dec = tokenizer(batch['output'], max_length=MAX_OUT, truncation=True, padding='max_length')
    enc['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in dec['input_ids']
    ]
    return enc

data = ds.map(tokenize, batched=True, remove_columns=['input', 'output'])
print(f'Train: {len(data["train"]):,}  |  Validation: {len(data["validation"]):,}')

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/14078 [00:00<?, ? examples/s]

Map:   0%|          | 0/1565 [00:00<?, ? examples/s]

Train: 14,078  |  Validation: 1,565


In [10]:
trainer = Seq2SeqTrainer(
    model         = model,
    args          = Seq2SeqTrainingArguments(
        output_dir                  = 'model',
        num_train_epochs            = 15,
        per_device_train_batch_size = 12,
        per_device_eval_batch_size  = 24,
        learning_rate               = 2e-4,
        warmup_steps                = 300,   # ramp LR from 0 for first 300 steps to avoid early instability
        weight_decay                = 0.01,  # L2 regularisation to discourage large weights
        eval_strategy               = 'steps',
        eval_steps                  = 500,
        save_strategy               = 'steps',
        save_steps                  = 500,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        greater_is_better           = False,
        predict_with_generate       = False,
        fp16                        = True,  # half-precision arithmetic halves VRAM usage
        logging_steps               = 100,
        report_to                   = 'none',
    ),
    train_dataset    = data['train'],
    eval_dataset     = data['validation'],
    processing_class = tokenizer,
    data_collator    = DataCollatorForSeq2Seq(tokenizer, model, padding=True, return_tensors='pt'),
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=5)],
)

print('Starting training. Expected: 45–90 min on A100.')
print('Validation Loss should decrease, then plateau. Training stops automatically.')
trainer.train()
print('\nTraining complete.')

Starting training. Expected: 45–90 min on A100.
Validation Loss should decrease, then plateau. Training stops automatically.


Step,Training Loss,Validation Loss
500,1.422738,1.136634
1000,1.068896,0.888234
1500,0.984717,0.784301
2000,1.032506,0.963577
2500,2.695053,1.993396
3000,2.669670,1.989180
3500,2.681105,1.985139
4000,2.657481,1.981282



Training complete.


---
## 5. Evaluation

We evaluate on held-out recipes the model never saw during training. We report three metrics:

- **Structural validity** — does the output contain both an `Ingredients` and a `Method` field?
- **Ingredient overlap** — does at least one generated ingredient name appear in the reference recipe? This is a loose proxy for coherence.
- **Exact match** — does the output match the reference verbatim? We expect this to be low - a high valuewould  suggests overfitting

Exact match being very low is actually expected and desired — we want the model to generate *new* recipes, not memorise old ones.

In [11]:
import torch, re

model.eval()
device = next(model.parameters()).device

def _predict(prompt, max_tokens=200):
    """Run beam search (4 beams) on a text prompt and return the decoded output.
    no_repeat_ngram_size=3 prevents repetitive outputs like 'gin, gin, gin'."""
    inp = tokenizer(prompt, return_tensors='pt', max_length=160, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_tokens,
                             num_beams=4, early_stopping=True, no_repeat_ngram_size=3)
    return tokenizer.decode(out[0], skip_special_tokens=True)

def _field(text, label):
    """Extract the value of a pipe-delimited field like 'Ingredients: ...'."""
    m = re.search(rf'{label}:\s*(.+?)(?:\s*\||$)', text, re.DOTALL)
    return m.group(1).strip() if m else ''

eval_set = test_pairs[:200]
n_total = n_exact = n_has_ings = n_has_method = n_ing_overlap = 0

for ex in eval_set:
    pred, target = _predict(ex['input']), ex['output']
    n_total += 1
    if pred.strip().lower() == target.strip().lower():
        n_exact += 1
    if _field(pred, 'Ingredients'):
        n_has_ings += 1
    if _field(pred, 'Method'):
        n_has_method += 1
    pred_ings   = _field(pred,   'Ingredients').lower()
    target_ings = _field(target, 'Ingredients').lower()
    ref_names   = {p.strip() for p in re.split(r',\s*', target_ings) if p.strip()}
    if any(name in pred_ings for name in ref_names if len(name) > 4):
        n_ing_overlap += 1

print(f'\n{"=" * 55}')
print(f'Evaluation on {n_total} held-out recipes')
print(f'{"=" * 55}')
print(f'  Has Ingredients field : {n_has_ings/n_total:5.1%}')
print(f'  Has Method field      : {n_has_method/n_total:5.1%}')
print(f'  ≥1 ingredient overlap : {n_ing_overlap/n_total:5.1%}')
print(f'  Exact match (<1%=good): {n_exact/n_total:5.1%}')

print('\nSample predictions:')
for ex in eval_set[:3]:
    pred = _predict(ex['input'])
    print(f'\n  Seed  : {ex["input"]}')
    print(f'  Target: {ex["output"][:100]}...')
    print(f'  Pred  : {pred[:100]}...')


Evaluation on 200 held-out recipes
  Has Ingredients field : 100.0%
  Has Method field      : 100.0%
  ≥1 ingredient overlap : 19.5%
  Exact match (<1%=good):  0.0%

Sample predictions:

  Seed  : create cocktail with: cognac [fruity floral oak]
  Target: Ingredients: 5 cl cognac, 3 cl cocchi americano bianco, 1 cl lemon juice (freshly squeezed), 2 cl du...
  Pred  : Ingredients: 4.5 cl rutte dry gin, 1.5 ml cognac, 2 dash angostura aromatic bitters, 1 dash peychaud...

  Seed  : create cocktail with: monin grenadine syrup [sweet pomegranate fruity]
  Target: Ingredients: 4 cl licor 43 original liqueur, 10 cl milk (whole milk/full 3-4% fat), 1 cl monin grena...
  Pred  : Ingredients: 4.5 cl rutte dry gin, 2.25 ml monin grenadine syrup, 1.5 l sugar syrup (65.0°brix, 2 su...

  Seed  : create cocktail with: grapefruit juice (pink) [bitter citrus tart]
  Target: Ingredients: 5 cl italicus liqueur, 12 cl grapefruit juice (pink) | Method: STIR all ingredients wit...
  Pred  : Ingredients: 

---
## 6. Save to Hugging Face Hub

Before saving, we **merge** the LoRA adapter matrices back into the base T5 weights. This produces a standard `T5ForConditionalGeneration` model that can be loaded with a plain `from_pretrained()` call — no PEFT library required by anyone who wants to use it later.

In [12]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '69b585df9e9446423f0f369a', 'name': 'jovinkk', 'fullname': 'Jo Ko', 'email': 'jovin.k@hotmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1775001600, 'isPro': False, 'avatarUrl': '/avatars/35f15bea0720713ad472f6492eb971d1.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Cocktail', 'role': 'write', 'createdAt': '2026-03-14T18:59:01.044Z'}}}


In [13]:
print('Merging LoRA weights into base model ...')
merged = model.merge_and_unload()  # folds LoRA deltas into the original weight matrices

print(f'Pushing to https://huggingface.co/{HF_REPO} ...')
merged.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f'\nDone. Load your model any time with:')
print(f'  from transformers import T5ForConditionalGeneration, T5Tokenizer')
print(f'  model     = T5ForConditionalGeneration.from_pretrained("{HF_REPO}")')
print(f'  tokenizer = T5Tokenizer.from_pretrained("{HF_REPO}")')

Merging LoRA weights into base model ...
Pushing to https://huggingface.co/jovinkk/cocktail-creator-t5 ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...u_opwzc/model.safetensors:   4%|4         | 36.9MB /  892MB            

README.md: 0.00B [00:00, ?B/s]


Done. Load your model any time with:
  from transformers import T5ForConditionalGeneration, T5Tokenizer
  model     = T5ForConditionalGeneration.from_pretrained("jovinkk/cocktail-creator-t5")
  tokenizer = T5Tokenizer.from_pretrained("jovinkk/cocktail-creator-t5")


---
## 7. Interactive App

The app runs entirely inside the cell output. A Python callback is registered via `google.colab.kernel.invokeFunction` and called directly by the JavaScript frontend — no external web server or ngrok needed.

Type any ingredient names (quantities are optional and are ignored — the model predicts all quantities), choose an optional style, and click **Create Cocktail**.

In [17]:
import re, os, json, torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from google.colab import output as colab_output
from IPython.display import display, HTML

# Load from HF Hub (works even after a session reset — no Drive mount needed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading model from {HF_REPO} on {device} ...')
_tok   = T5Tokenizer.from_pretrained(HF_REPO)
_model = T5ForConditionalGeneration.from_pretrained(HF_REPO).to(device).eval()
print('Model loaded.')

# Output parsers ----------------------------------------------------------------
def _field(text, label):
    m = re.search(rf'{label}:\s*(.+?)(?:\s*\||$)', text, re.DOTALL)
    return m.group(1).strip() if m else ''

# Expanded unit list covering non-standard abbreviations in the training data
_ING_RE = re.compile(
    r'^([\d\u00bc\u00bd\u00be\u2153\u2154]+(?:[\/\.]\d+)?\s*'
    r'(?:oz|cl|ml|tsp|tbsp|tl|ts|cs|ct|c(?=\s)|'
    r'dash(?:es)?|splash|barspoon|shot|drop|part|parts|measure)s?'
    r'(?:\s+of)?)\s*(.+)$',
    re.IGNORECASE,
)

# Patterns to strip from ingredient names
_CLEAN_RE = [
    re.compile(r'\(\s*\d+[\d\.\s]*%?\s*\)'),          # (40%), (65.0%)
    re.compile(r'\(\s*[\d\.]+\s*°?\s*\w+\s*\)'),       # (65.0°Brix)
    re.compile(r'\(\s*optional\s*\)', re.IGNORECASE),   # (Optional)
    re.compile(r'\b\d+\s+to\s+\d+\b.*$', re.IGNORECASE), # "2 Sugar To 1 Water Rich Syrup"
    re.compile(r'\s{2,}'),                              # double spaces
]

def _clean_name(name: str) -> str:
    """Strip brand annotations, percentages, and noise from ingredient names."""
    for pattern in _CLEAN_RE:
        name = pattern.sub('', name)
    return name.strip().strip(')').strip()

def _parse_output_ingredients(raw):
    ings = []
    for part in re.split(r',\s*', raw):
        part = part.strip()
        if not part:
            continue
        m = _ING_RE.match(part)
        if m:
            ings.append({
                'measure': m.group(1).strip(),
                'name':    _clean_name(m.group(2))
            })
        else:
            ings.append({'measure': '', 'name': _clean_name(part)})
    # Drop any ingredients where the name ended up empty after cleaning
    return [i for i in ings if i['name']]

def parse_recipe_output(text):
    method_raw = _field(text, 'Method') or ''
    # Take only the first sentence and capitalise properly
    method_clean = re.split(r'(?<=[.!?])\s+', method_raw)[0].strip()
    return {
        'ingredients': _parse_output_ingredients(_field(text, 'Ingredients')),
        'method':      method_clean,
        'glass':       _field(text, 'Glass') or ''
    }

# Default quantities for any seed not generated by the model --------------------
_DEFAULT_QTY = [
    (['whiskey','whisky','bourbon','rye','gin','vodka',
      'rum','tequila','mezcal','brandy','cognac','pisco'], '4.5 cl'),
    (['liqueur','triple sec','cointreau','campari','aperol',
      'amaretto','kahlua','chartreuse','curacao','vermouth'], '2.25 cl'),
    (['lemon juice','lime juice','orange juice','grapefruit juice'], '2.25 cl'),
    (['syrup','grenadine','orgeat','falernum','honey'], '1.5 cl'),
    (['cream','coconut cream','half and half'], '3 cl'),
    (['bitters'], '2 dash'),
    (['soda','tonic','ginger beer','ginger ale','cola'], '6 cl'),
    (['juice'], '3 cl'),
]

def _qty_for(name):
    nl = name.lower()
    for keywords, qty in _DEFAULT_QTY:
        if any(kw in nl for kw in keywords):
            return qty
    return '4.5 cl'

# Inference callback (called by JavaScript frontend) ----------------------------
def generate_recipe(ingredients_text, style, random_seed=None):
    try:
        seed_names, seeds = [], []
        for raw in re.split(r',\s*', ingredients_text.strip()):
            raw = raw.strip()
            if not raw:
                continue
            # Strip any quantity the user may have typed
            m    = re.match(r'^([\d\u00bc\u00bd\u00be\u2153\u2154]+(?:[\/\.]\d+)?\s*'
                            r'(?:oz|cl|ml|tsp|tbsp|dash(?:es)?|splash|barspoon|shot)s?'
                            r'(?:\s+of)?)\s*(.+)$', raw, re.IGNORECASE)
            name = m.group(2).strip().lower() if m else raw.lower()
            seed_names.append(name)
            seeds.append(f"{name} [{' '.join(get_flavor(name)[:3])}]")

        if not seeds:
            return json.dumps({'error': 'No valid ingredients provided'})

        prompt = 'create cocktail with: ' + '; '.join(seeds)
        if style.strip():
            prompt += f' | style: {style.strip()}'

        if random_seed is not None:
            torch.manual_seed(int(random_seed))

        inp = _tok(prompt, return_tensors='pt', max_length=160, truncation=True).to(device)
        with torch.no_grad():
            out = _model.generate(
                **inp, max_new_tokens=200,
                do_sample=True, temperature=0.85, top_p=0.92,
                no_repeat_ngram_size=3,
            )
        result = parse_recipe_output(_tok.decode(out[0], skip_special_tokens=True))

        # Inject any seeds the model dropped (models occasionally omit a seed ingredient)
        generated_text = ' '.join(i['name'].lower() for i in result['ingredients'])
        for seed_name in seed_names:
            key_word = seed_name.split()[0] if ' ' in seed_name else seed_name
            if key_word not in generated_text and seed_name not in generated_text:
                result['ingredients'].insert(0, {'measure': _qty_for(seed_name), 'name': seed_name})

        return json.dumps(result, ensure_ascii=False)
    except Exception as e:
        import traceback; traceback.print_exc()
        return json.dumps({'error': str(e)})


colab_output.register_callback('generate_recipe', generate_recipe)
print('Callback registered.')

Loading model from YOUR_HF_USERNAME/cocktail-creator-t5 on cuda ...


RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69b5bb65-616cea9315f128094f307f39;b281c478-3af3-4ea8-9ae3-4d1a0ef07453)

Repository Not Found for url: https://huggingface.co/api/models/YOUR_HF_USERNAME/cocktail-creator-t5/tree/main/additional_chat_templates?recursive=false&expand=false.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication

In [15]:
APP_HTML = """
<style>
  .cc-wrap{font-family:'Segoe UI',Arial,sans-serif;max-width:680px;margin:0 auto;padding:24px;}
  h2{margin:0 0 4px;font-size:1.6em;color:#1a1a1a;}  .subtitle{color:#666;margin:0 0 20px;font-size:.95em;}
  .row{display:flex;gap:12px;margin-bottom:12px;flex-wrap:wrap;}
  input,select{flex:1;min-width:0;padding:10px 14px;border:1.5px solid #ddd;border-radius:8px;
               font-size:1em;outline:none;transition:border .2s;}
  input:focus,select:focus{border-color:#4a90d9;}
  .btn{padding:11px 28px;background:#1a6b72;color:#fff;border:none;border-radius:8px;
       font-size:1em;font-weight:600;cursor:pointer;transition:background .2s;}
  .btn:hover{background:#135558;} .btn:disabled{background:#999;cursor:default;}
  .card{background:#f8fffe;border:1.5px solid #b2d8da;border-radius:12px;padding:22px;
        margin-top:20px;display:none;}
  .card-title{font-size:1.3em;font-weight:700;color:#1a6b72;margin-bottom:16px;}
  .ing-row{display:flex;align-items:baseline;gap:10px;padding:6px 0;
            border-bottom:1px solid #e8f4f5;}
  .ing-qty{font-weight:600;min-width:70px;color:#1a6b72;font-size:.95em;}
  .ing-name{color:#333;text-transform:capitalize;}
  .meta{display:flex;gap:20px;margin-top:14px;flex-wrap:wrap;}
  .meta-pill{background:#e8f4f5;border-radius:20px;padding:5px 14px;font-size:.88em;color:#1a6b72;font-weight:600;}
  .err{color:#c0392b;background:#fdf0ef;border:1px solid #f5c6c6;padding:10px 14px;
       border-radius:8px;margin-top:12px;display:none;}
  .hint{font-size:.82em;color:#999;margin-top:6px;}
</style>
<div class="cc-wrap">
  <h2>🍹 Cocktail Creator</h2>
  <p class="subtitle">Type one or more ingredients, then hit Create.</p>
  <div class="row">
    <input id="ings" placeholder="e.g. bourbon, lemon juice, honey" />
    <select id="style">
      <option value="">Any style</option>
      <option>sour</option><option>highball</option><option>tiki</option>
      <option>spritz</option><option>stirred</option><option>creamy</option>
    </select>
  </div>
  <p class="hint">Quantities are ignored — the model predicts them for you.</p>
  <div class="row">
    <button class="btn" id="btn" onclick="createCocktail()">Create Cocktail</button>
    <button class="btn" id="rbtn" style="background:#555;display:none" onclick="createCocktail(true)">↺ Regenerate</button>
  </div>
  <div class="err" id="err"></div>
  <div class="card" id="card">
    <div class="card-title" id="card-title">Your Cocktail</div>
    <div id="ings-out"></div>
    <div class="meta" id="meta"></div>
  </div>
</div>
<script>
let _seed = 0;
async function createCocktail(regen=false) {
  const ings  = document.getElementById('ings').value.trim();
  const style = document.getElementById('style').value;
  if (!ings) { showErr('Please enter at least one ingredient.'); return; }
  if (regen) _seed++;
  document.getElementById('btn').disabled  = true;
  document.getElementById('btn').textContent = 'Creating…';
  document.getElementById('err').style.display  = 'none';
  document.getElementById('card').style.display = 'none';
  try {
    const raw = await google.colab.kernel.invokeFunction('generate_recipe', [ings, style, _seed], {});
    const res = JSON.parse(raw.data['text/plain'].replace(/^'|'$/g,'').replace(/\\'/g,"'"));
    if (res.error) { showErr(res.error); return; }
    const ingsHtml = (res.ingredients || []).map(i =>
      `<div class="ing-row"><span class="ing-qty">${i.measure||''}</span><span class="ing-name">${i.name||''}</span></div>`
    ).join('');
    document.getElementById('ings-out').innerHTML = ingsHtml;
    const meta = [];
    if (res.method) meta.push(`<span class="meta-pill">🥃 ${res.method}</span>`);
    if (res.glass)  meta.push(`<span class="meta-pill">🍸 ${res.glass}</span>`);
    document.getElementById('meta').innerHTML = meta.join('');
    document.getElementById('card').style.display = 'block';
    document.getElementById('rbtn').style.display = '';
  } catch(e) { showErr(e.toString()); }
  finally {
    document.getElementById('btn').disabled = false;
    document.getElementById('btn').textContent = 'Create Cocktail';
  }
}
function showErr(msg) {
  const el = document.getElementById('err');
  el.textContent = msg; el.style.display = 'block';
}
document.getElementById('ings').addEventListener('keydown', e => { if (e.key==='Enter') createCocktail(); });
</script>
"""
display(HTML(APP_HTML))

---
## 8. Quick Resume After Disconnect

Colab sessions reset after ~90 minutes of inactivity. Because the model lives on Hugging Face Hub, one can use it withou the need to retrain. Just run this cell to reinstall packages, then re-run **Section 2** (FlavorDB) and **Section 7** (the app).

Total resume time is roughly 2–3 minutes.

In [16]:
!pip install -q peft accelerate sentencepiece datasets rapidfuzz huggingface_hub

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
HF_REPO  = 'jovinkk/cocktail-creator-t5'  # <-- same as above

login(token=HF_TOKEN)
print('Ready. Now re-run Section 2 (FlavorDB) and Section 7 (App).')

Ready. Now re-run Section 2 (FlavorDB) and Section 7 (App).
